In [1]:
!pip install torch_geometric

In [31]:
!pip install torchinfo

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
import torch.optim as optim
from tqdm.notebook import tqdm
import pytorch_model_summary
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader as PyTorchDataLoader
from torch.utils.data import TensorDataset
import pickle
import os
import random
import copy


from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_dense_adj
from torch.utils.data import random_split, Subset
from torchinfo import summary

# Generating Data

https://git.litislab.fr/bgauzere/py-graph/-/tree/v0.2/datasets/NCI1?ref_type=heads

graph Dataset NCI1을 이용

Chemical Molescular를 그래프의 형태로 feature로, discrete label을 label로 하는 dataset

대칭성을 이용한 MLP와 단순 MLP를 비교

In [2]:
device = torch.device('mps' if torch.backends.mps.is_available() and torch.backends.mps.is_built() else 'cpu')

In [3]:
dataset = TUDataset(root='/tmp/NCI1', name='NCI1')
max_nodes = 50
node_features = dataset.num_node_features

Processing...
Done!


In [4]:
class PermutedDataset(Dataset):

    def __init__(self, base_dataset, max_nodes):

        self.base_dataset = base_dataset
        self.max_nodes = max_nodes

        self.permutations = []

        for data in base_dataset:

            num_nodes = min(data.x.size(0), max_nodes)
            perm_real = torch.randperm(num_nodes)
            perm_full = torch.cat([
                perm_real,
                torch.arange(num_nodes, max_nodes)
            ])

            self.permutations.append(perm_full)

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        data = self.base_dataset[idx]
        perm = self.permutations[idx]

        adj = to_dense_adj(
            data.edge_index,
            max_num_nodes=self.max_nodes
        )[0]

        x = torch.zeros(
            (self.max_nodes, node_features)
        )

        num_nodes = min(data.x.size(0), self.max_nodes)

        x[:num_nodes] = data.x[:num_nodes]

        ####################################################
        # permutation 적용
        ####################################################

        adj = adj[perm][:, perm]
        x = x[perm]

        return adj, x, data.y

In [5]:
permuted_dataset = PermutedDataset(dataset, max_nodes)

In [6]:
def collate_fn(batch):

    adjs = []
    xs = []
    ys = []

    for adj, x, y in batch:

        adjs.append(adj)
        xs.append(x)
        ys.append(y)

    return (
        torch.stack(adjs).to(device),
        torch.stack(xs).to(device),
        torch.stack(ys).to(device)
    )

In [7]:
train_size = int(0.8 * len(permuted_dataset))
val_size = int(0.1 * len(permuted_dataset))
test_size = len(permuted_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(permuted_dataset, [train_size, val_size, test_size])

train_loader = PyTorchDataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)
val_loader = PyTorchDataLoader(val_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn)
test_loader = PyTorchDataLoader(test_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn)

In [8]:
max_nodes = 50
node_features = dataset.num_node_features
num_classes = dataset.num_classes

# Defining

### model

In [9]:
class symmetricGraphLayer(nn.Module):
    """
    Most general linear S_n-equivariant layer
    for

        (A, X) -> Y

    where

        A : (B, N, N)
        X : (B, N, Fin)
        Y : (B, N, Fout)

    satisfying

        f(PAP^T, PX) = P f(A, X)

    for every permutation matrix P.

    --------------------------------------------------------
    Basis construction
    --------------------------------------------------------

    We classify all admissible linear operators using
    equality patterns of indices (i,j,k).

    Bell number B_3 = 5
    => exactly 5 basis operators.

    Basis:

        B1 : i=j=k
        B2 : i=j!=k
        B3 : i=k!=j
        B4 : j=k!=i
        B5 : all distinct

    Any linear S_n-equivariant operator is a linear
    combination of these 5 basis operators.
    """

    def __init__(self, in_dim, out_dim, bias=True):
        super().__init__()

        self.W1 = nn.Linear(in_dim, out_dim, bias=False)
        self.W2 = nn.Linear(in_dim, out_dim, bias=False)
        self.W3 = nn.Linear(in_dim, out_dim, bias=False)
        self.W4 = nn.Linear(in_dim, out_dim, bias=False)
        self.W5 = nn.Linear(in_dim, out_dim, bias=False)

        if bias:
            self.bias = nn.Parameter(torch.zeros(out_dim))
        else:
            self.register_parameter("bias", None)   

    def forward(self, A, X):

        B, N, _ = A.shape

        ############################################
        # Basic quantities
        ############################################

        # diagonal(A)
        diagA = torch.diagonal(A, dim1=-2, dim2=-1)
        # (B,N)

        diagA_col = diagA.unsqueeze(-1)
        # (B,N,1)

        # degree
        deg = A.sum(dim=-1, keepdim=True)
        # (B,N,1)

        ############################################
        # Basis 1
        #
        # i = j = k
        #
        # diag(A)_i * X_i
        ############################################

        B1 = diagA_col * X

        ############################################
        # Basis 2
        #
        # i = j != k
        #
        # sum_{k!=i} A_ik X_k
        #
        # = AX - diag(A)X
        ############################################

        AX = torch.matmul(A, X)

        B2 = AX - B1

        ############################################
        # Basis 3
        #
        # i = k != j
        #
        # sum_{j!=i} A_ji X_i
        #
        # undirected:
        # (deg_i - A_ii) X_i
        ############################################

        B3 = (deg - diagA_col) * X

        ############################################
        # Basis 4
        #
        # j = k != i
        #
        # sum_{j!=i} A_jj X_j
        ############################################

        global_diag = (diagA_col * X).sum(dim=1, keepdim=True)
        # (B,1,F)

        B4 = global_diag.expand(-1, N, -1) - B1

        ############################################
        # Basis 5
        #
        # i,j,k all distinct
        #
        # remaining interaction
        ############################################

        global_AX = AX.sum(dim=1, keepdim=True)
        # (B,1,F)

        total_expand = global_AX.expand(-1, N, -1)

        B5 = total_expand - B1 - B2 - B3 - B4

        ############################################
        # Linear combination of basis operators
        ############################################

        Y = (
            self.W1(B1)
            + self.W2(B2)
            + self.W3(B3)
            + self.W4(B4)
            + self.W5(B5)
        )

        if self.bias is not None:
            Y = Y + self.bias

        return Y

In [10]:
class AttentionPooling(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        # 각 노드의 중요도를 스칼라(1차원) 점수로 변환하는 신경망
        self.attn_net = nn.Sequential(
            nn.Linear(in_dim, in_dim // 2),
            nn.Tanh(),
            nn.Linear(in_dim // 2, 1)
        )
        
    def forward(self, X):
        # X shape: [batch_size, 50, current_dim]
        
        # 1. 각 노드별 중요도 점수 계산 -> [batch_size, 50, 1]
        scores = self.attn_net(X)
        
        # 2. 노드 방향(dim=1)으로 Softmax를 취해 가중치(확률 분포) 생성 -> [batch_size, 50, 1]
        # 제로 패딩된 노드가 있을 경우 가중치가 분산되는 것을 막기 위해 Softmax를 취합니다.
        weights = F.softmax(scores, dim=1)
        
        # 3. 가중치를 각 노드 피처에 곱하고 합산 (Weighted Sum) -> [batch_size, current_dim]
        graph_feat = torch.sum(weights * X, dim=1)
        
        return graph_feat, weights

class symmetricGraphMLP(nn.Module):
    def __init__(self, in_dim, hidden_size, out_dim):
        super().__init__()
        self.layers = nn.ModuleList()
        self.norms = nn.ModuleList()        # [추가] 각 레이어별 LayerNorm을 담을 리스트
        
        current_dim = in_dim
        for h_size in hidden_size:
            # 1. 대칭성 그래프 레이어 추가
            self.layers.append(symmetricGraphLayer(in_dim=current_dim, out_dim=h_size))
            self.norms.append(nn.LayerNorm(h_size))
                
            current_dim = h_size

        # 정보 보존하기 위한 가중치 합 풀링 적용
        self.pool = AttentionPooling(in_dim=current_dim)

        # 최종 분류기
        self.classifier = nn.Sequential(
            nn.Linear(current_dim, current_dim),
            nn.LeakyReLU(0.01),
            nn.Linear(current_dim, out_dim)
        )

    def forward(self, A, X):
        for layer, norm in zip(self.layers, self.norms):
            out = layer(A, X)
            out = norm(out)
            X = F.leaky_relu(out, negative_slope=0.01)
        graph_feat, attn_weights = self.pool(X)
        
        # 3. 분류기 통과 -> [batch_size, out_dim]
        return self.classifier(graph_feat)

In [12]:
class vanillaGraphMLP(nn.Module):
    def __init__(self, max_nodes, in_dim, hidden_size, out_dim):
        super().__init__()
        self.max_nodes = max_nodes
        
        # 1. 입력 데이터를 일렬로 펼쳤을 때의 총 차원 계산
        # 인접 행렬(max_nodes * max_nodes) + 노드 특징(max_nodes * in_dim)
        # NCI1의 예시 (max_nodes=50, in_dim=37) 일 때: 2500 + 1850 = 4350 차원
        input_flat_dim = max_nodes * max_nodes + max_nodes * in_dim
        
        self.layers = nn.ModuleList()
        current_dim = input_flat_dim
        
        # 2. 대칭 모델과 동일한 hidden_size 깊이와 너비로 일반 Linear 레이어 구축
        for h_size in hidden_size:
            self.layers.append(nn.Linear(current_dim, h_size))
            current_dim = h_size
            
        # 3. 최종 분류기 (대칭 모델과 동일한 구조 적용)
        self.classifier = nn.Sequential(
            nn.Linear(current_dim, current_dim),
            nn.LeakyReLU(0.01),
            nn.Linear(current_dim, out_dim)
        )

    def forward(self, A, X):
        # A shape: [batch_size, 50, 50] -> [batch_size, 2500]
        # X shape: [batch_size, 50, in_dim] -> [batch_size, 50 * in_dim]
        
        # 1. 그래프 구조를 완전히 무시하고 일렬로 평탄화(Flatten)
        A_flat = A.view(A.size(0), -1)
        X_flat = X.view(X.size(0), -1)
        
        # 2. 두 텐서를 결합하여 하나의 거대한 벡터로 변환
        # feat shape: [batch_size, input_flat_dim]
        feat = torch.cat([A_flat, X_flat], dim=-1)
        
        # 3. 일반 MLP 은닉층 통과
        for layer in self.layers:
            feat = layer(feat)
            feat = F.leaky_relu(feat, negative_slope=0.01)
            
        # 4. 최종 분류기 통과 -> [batch_size, out_dim]
        return self.classifier(feat)

### functions

In [11]:
def train(model, train_loader, val_loader=None, epochs=100, lr=1e-3, early_stop=False, patience=-1, augment = False):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    # NCI1 데이터셋은 그래프 분류 태스크이므로 CrossEntropyLoss를 사용합니다.
    criterion = nn.CrossEntropyLoss()

    train_log = []
    valid_log = []
    best_loss = float('inf')
    best_model_state = None
    patience_counter = 0

    bar = tqdm(range(epochs), desc='Training')
    for epoch in bar:
        model.train()
        epoch_loss = 0
        correct = 0
        total = 0
        
        # collate_fn을 통해 생성된 batch_A, batch_X, batch_Y 사용
        for batch_A, batch_X, batch_Y in train_loader:
            if augment:
                N = batch_A.shape[1] # max_nodes = 50  
                p_indices = torch.randperm(N, device=batch_A.device)
                
                batch_X_permuted = batch_X[:, p_indices, :]
                batch_A_permuted = batch_A[:, p_indices, :][:, :, p_indices]
            else:
                batch_X_permuted = batch_X
                batch_A_permuted = batch_A
            optimizer.zero_grad()
            
            # 모델 순전파 (A와 X를 입력으로 받음)
            pred = model(batch_A_permuted, batch_X_permuted)
            loss = criterion(pred, batch_Y)
            
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item() * batch_A_permuted.size(0)
            _, predicted = pred.max(1)
            total += batch_Y.size(0)
            correct += predicted.eq(batch_Y).sum().item()
        
        avg_train_loss = epoch_loss / total
        train_acc = correct / total
        train_log.append(avg_train_loss)

        if early_stop and val_loader is not None:
            model.eval()
            epoch_loss_valid = 0
            val_correct = 0
            val_total = 0
            
            with torch.no_grad():
                for batch_A, batch_X, batch_Y in val_loader:
                    N = batch_A.shape[1] # max_nodes = 50  
                    p_indices = torch.randperm(N, device=batch_A.device)
                    
                    batch_X_permuted = batch_X[:, p_indices, :]
                    batch_A_permuted = batch_A[:, p_indices, :][:, :, p_indices]

                    pred = model(batch_A_permuted, batch_X_permuted)
                    loss = criterion(pred, batch_Y)
                    epoch_loss_valid += loss.item() * batch_A_permuted.size(0)
                    
                    _, predicted = pred.max(1)
                    val_total += batch_Y.size(0)
                    val_correct += predicted.eq(batch_Y).sum().item()
                    
            avg_loss_valid = epoch_loss_valid / val_total
            val_acc = val_correct / val_total
            valid_log.append(avg_loss_valid)

            bar.set_description(f'Loss: {avg_train_loss:.4f} | Acc: {train_acc*100:.1f}% | Val Loss: {avg_loss_valid:.4f} | Val Acc: {val_acc*100:.1f}%')

            # Early Stopping 조건 체크
            if avg_loss_valid < best_loss:
                best_loss = avg_loss_valid
                best_model_state = model.state_dict()
                patience_counter = 0
            else:
                patience_counter += 1
            
            if patience > 0 and patience_counter >= patience:
                print(f"Early stopping triggered at epoch {epoch+1}")
                break
        else:
            bar.set_description(f'Loss: {avg_train_loss:.4f} | Acc: {train_acc*100:.1f}%')

    if early_stop and best_model_state is not None:
        model.load_state_dict(best_model_state)
        
    return {
        'train_loss': train_log,
        'valid_loss': valid_log if early_stop else []
    }

@torch.no_grad()
def symmetry_error_graph(model, loader, repeat=5):
    """
    노드의 순서를 무작위로 섞었을 때(Permutation) 모델의 출력(Logit) 변화량이 얼마나 발생하는지 측정합니다.
    Symmetric 모델은 이 오차가 0에 수렴해야 하며, 단순 MLP는 큰 오차가 발생합니다.
    """
    model.eval()
    error_list = []
    
    for batch_A, batch_X, _ in loader:
        N = batch_A.shape[1] # max_nodes = 50
        
        # 1. 원본 그래프 데이터 예측 결과 (Logits)
        y_base = model(batch_A, batch_X)
        
        for _ in range(repeat):
            # 무작위 노드 순서 인덱스 생성
            p_indices = torch.randperm(N, device=batch_A.device)
            
            # 2. 노드 피처 치환 (P * X)
            X_permuted = batch_X[:, p_indices, :]
            
            # 3. 인접 행렬 치환 (P * A * P^T) -> 행과 열을 모두 동일한 인덱스로 셔플
            A_permuted = batch_A[:, p_indices, :][:, :, p_indices]
            
            # 4. 변형된 그래프 데이터 예측 결과
            y_permuted = model(A_permuted, X_permuted)
            
            # 원본 결과와 변형 결과 사이의 Mean Squared Error 계산
            mse = torch.mean((y_base - y_permuted) ** 2)
            error_list.append(mse.item())
            
    return np.mean(error_list)

@torch.no_grad()
def evaluate_graph(name, model, loader):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_A, batch_X, batch_Y in loader:
        N = batch_A.shape[1] # max_nodes = 50
        p_indices = torch.randperm(N, device=batch_A.device)
        
        batch_X_permuted = batch_X[:, p_indices, :]
        batch_A_permuted = batch_A[:, p_indices, :][:, :, p_indices]
        
        pred = model(batch_A_permuted, batch_X_permuted)
        loss = criterion(pred, batch_Y)
        
        total_loss += loss.item() * batch_A.size(0)
        _, predicted = pred.max(1)
        total += batch_Y.size(0)
        correct += predicted.eq(batch_Y).sum().item()
        
    avg_loss = total_loss / total
    accuracy = correct / total
    
    sym_err = symmetry_error_graph(model, loader, repeat=5)
    
    print(f'[{name} Evaluation Result - Permuted Data]')
    print(f'  Loss: {avg_loss:.6f}')
    print(f'  Accuracy: {accuracy * 100:.2f}%')
    print(f'  Symmetry Error: {sym_err:.6e}\n')
    
    return avg_loss, accuracy, sym_err

# Running

## Model1

In [14]:
hidden_dims = [128, 128, 64] # 비교 레이어 채널 크기

# [실험군] 가중치 합 어텐션 기반 대칭 그래프 신경망
symmetric_model = symmetricGraphMLP(
    in_dim=node_features, 
    hidden_size=hidden_dims, 
    out_dim=num_classes
).to(device)

# [대조군] 구조를 완전히 일렬로 펴서 학습하는 일반 MLP
vanilla_model = vanillaGraphMLP(
    max_nodes=max_nodes, 
    in_dim=node_features, 
    hidden_size=hidden_dims, 
    out_dim=num_classes
).to(device)

In [16]:
print("==================== [1] Symmetric Graph MLP Training ====================")
shared_log = train(
    model=symmetric_model, 
    train_loader=train_loader, 
    val_loader=val_loader, 
    epochs=50, 
    lr=1e-3, 
    early_stop=True, 
    patience=10
)

print("\n==================== [2] Vanilla Graph MLP Training ====================")
vanilla_log = train(
    model=vanilla_model,
    train_loader=train_loader, 
    val_loader=val_loader, 
    epochs=50, 
    lr=1e-3, 
    early_stop=True, 
    patience=10
)

==================== [1] Symmetric Graph MLP Training ====================


Training:   0%|          | 0/50 [00:00<?, ?it/s]

Early stopping triggered at epoch 29

==================== [2] Vanilla Graph MLP Training ====================


Training:   0%|          | 0/50 [00:00<?, ?it/s]

Early stopping triggered at epoch 11


In [17]:
sym_loss, sym_acc, sym_err = evaluate_graph("Symmetric Model", symmetric_model, test_loader)
van_loss, van_acc, van_err = evaluate_graph("Vanilla Model", vanilla_model, test_loader)

[Symmetric Model Evaluation Result - Permuted Data]
  Loss: 0.603127
  Accuracy: 64.96%
  Symmetry Error: 4.805929e-15

[Vanilla Model Evaluation Result - Permuted Data]
  Loss: 7.257290
  Accuracy: 54.74%
  Symmetry Error: 1.048032e+02



## Data Generating

In [33]:

# 1. 결과 및 체크포인트를 저장할 디렉토리 생성
models_path = './models'
logs_path = './logs'
os.makedirs(models_path, exist_ok=True)
os.makedirs(logs_path, exist_ok=True)

# ====================================================================
# [수정] 전체 데이터셋의 인덱스를 미리 무작위로 섞어둡니다.
# 이렇게 해야 특정 data_size를 뽑았을 때 클래스가 한쪽으로 몰리지 않습니다.
# ====================================================================
all_indices = list(range(len(dataset)))
random.seed(42)      # 실험의 재현성을 위해 파이썬 내장 random 시드 고정
random.shuffle(all_indices)
# ====================================================================

# 2. Grid Search를 수행할 하이퍼파라미터 조건 정의
n_blocks_list = [3, 5, 7]
data_sizes_list = [500, 1500, len(dataset)]
hidden_dims_lst = [
    [64,  64,  16],
    [256, 256, 128, 64,  16],
    [256, 512, 256, 128, 128, 64, 16]
]

result = {}

for i in range(len(n_blocks_list)):
    n_block = n_blocks_list[i]
    curr_result = {}
    hidden_dims = hidden_dims_lst[i] 
    
    for data_size in data_sizes_list:
        print(f"\n" + "="*60)
        print(f"▶ [실험 시작] Blocks(깊이): {n_block} | Dataset Size: {data_size}")
        print("="*60)
        
        indices = all_indices[:data_size]
        subset_dataset = Subset(permuted_dataset, indices)
        
        train_size = int(0.8 * data_size)
        val_size = int(0.1 * data_size)
        test_size = data_size - train_size - val_size
        
        if train_size == 0 or val_size == 0 or test_size == 0:
            print(f"데이터 크기가 너무 작아 실험을 건너뜁니다. (Size: {data_size})")
            continue
            
        train_dataset, val_dataset, test_dataset = random_split(
            subset_dataset, [train_size, val_size, test_size],
            generator=torch.Generator().manual_seed(42)
        )
        
        # 3-3. PyTorch 순정 DataLoader 생성 (앞서 에러를 해결한 방식)
        train_loader = PyTorchDataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)
        val_loader = PyTorchDataLoader(val_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn)
        test_loader = PyTorchDataLoader(test_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn)
        
        # 3-4. [실험군] Symmetric 모델 선언 및 학습
        print(f"\n--- [1] Symmetric Graph MLP 학습 개시 ---")
        symmetric_model = symmetricGraphMLP(
            in_dim=node_features, 
            hidden_size=hidden_dims, 
            out_dim=num_classes
        ).to(device)
        
        shared_log = train(
            model=symmetric_model, 
            train_loader=train_loader, 
            val_loader=val_loader, 
            epochs=100, # Grid Search 속도를 위해 에포크 조정 가능
            lr=1e-3, 
            early_stop=True, 
            patience=5
        )
        
        # 3-5. [대조군] Vanilla MLP 모델 선언 및 학습
        print(f"\n--- [2] Vanilla Graph MLP  학습 개시 ---")
        vanilla_model = vanillaGraphMLP(
            max_nodes=max_nodes, 
            in_dim=node_features, 
            hidden_size=hidden_dims, 
            out_dim=num_classes
        ).to(device)
        
        vanilla_log = train(
            model=vanilla_model, 
            train_loader=train_loader, 
            val_loader=val_loader, 
            epochs=100, 
            lr=1e-3, 
            early_stop=True, 
            patience=5
        )

        print(f"\n--- [3] Symmetric Graph MLP (with Augmented Data) 학습 개시 ---")
        symmetric_model_augmented = symmetricGraphMLP(
            in_dim=node_features, 
            hidden_size=hidden_dims, 
            out_dim=num_classes
        ).to(device)
        
        shared_log = train(
            model=symmetric_model_augmented, 
            train_loader=train_loader, 
            val_loader=val_loader, 
            epochs=100, # Grid Search 속도를 위해 에포크 조정 가능
            lr=1e-3, 
            early_stop=True, 
            patience=5,
            augment = True
        )
        
        # 3-5. [대조군] Vanilla MLP 모델 선언 및 학습
        print(f"\n--- [4] Vanilla Graph MLP (with Augmented Data)  학습 개시 ---")
        vanilla_model_augmented = vanillaGraphMLP(
            max_nodes=max_nodes, 
            in_dim=node_features, 
            hidden_size=hidden_dims, 
            out_dim=num_classes
        ).to(device)
        
        vanilla_log = train(
            model=vanilla_model_augmented, 
            train_loader=train_loader, 
            val_loader=val_loader, 
            epochs=100, 
            lr=1e-3, 
            early_stop=True, 
            patience=5,
            augment = True
        )
        
        # 3-6. 최종 Test 데이터셋을 바탕으로 성능 및 대칭성 에러(Symmetry Error) 측정
        print(f"\n--- [5] 최종 성능 검증 및 Symmetry Error 비교 ---")
        sym_loss, sym_acc, sym_err = evaluate_graph("Symmetric Model", symmetric_model, test_loader)
        van_loss, van_acc, van_err = evaluate_graph("Vanilla Model", vanilla_model, test_loader)
        sym_aug_loss, sym_aug_acc, sym_aug_err = evaluate_graph("Symmetric Model(augmented)", symmetric_model_augmented, test_loader)
        van_aug_loss, van_aug_acc, van_aug_err = evaluate_graph("Vanilla Model(augmented)", vanilla_model_augmented, test_loader)
        
        # 3-7. 현재 조건의 결과를 딕셔너리에 저장
        curr_result[data_size] = {
            'symmetric': {'loss': sym_loss, 'acc': sym_acc, 'sym_error': sym_err},
            'vanilla': {'loss': van_loss, 'acc': van_acc, 'sym_error': van_err},
            'symmetric (data augmented)': {'loss': sym_aug_loss, 'acc': sym_aug_acc, 'sym_error': sym_aug_err},
            'vanilla (data augmented)': {'loss': van_aug_loss, 'acc': van_aug_acc, 'sym_error': van_aug_err}
        }
        
        # 3-8. 학습된 모델 가중치(체크포인트)와 개별 Loss 로그 중간 저장
        torch.save(symmetric_model.state_dict(), os.path.join(models_path, f'shared_model_{n_block}_{data_size}.pt'))
        torch.save(vanilla_model.state_dict(), os.path.join(models_path, f'vanilla_model_{n_block}_{data_size}.pt'))
        
        with open(os.path.join(logs_path, f'shared_log_{n_block}_{data_size}.pkl'), 'wb') as f:
            pickle.dump(shared_log, f)
        with open(os.path.join(logs_path, f'vanilla_log_{n_block}_{data_size}.pkl'), 'wb') as f:
            pickle.dump(vanilla_log, f)

    # 한 개의 블록 실험이 끝날 때마다 결과 누적
    result[n_block] = curr_result

# 4. 모든 Grid Search 과정이 완료된 최종 결과 데이터 통째로 저장
with open('./total_result.pkl', 'wb') as f:
    pickle.dump(result, f)
print("\n모든 Grid Search 실험 조건 완수 및 파일 저장 완료")


▶ [실험 시작] Blocks(깊이): 3 | Dataset Size: 500

--- [1] Symmetric Graph MLP 학습 개시 ---


Training:   0%|          | 0/100 [00:00<?, ?it/s]

Early stopping triggered at epoch 18

--- [2] Vanilla Graph MLP  학습 개시 ---


Training:   0%|          | 0/100 [00:00<?, ?it/s]

Early stopping triggered at epoch 6

--- [3] Symmetric Graph MLP (with Augmented Data) 학습 개시 ---


...

Early stopping triggered at epoch 6

--- [3] Symmetric Graph MLP (with Augmented Data) 학습 개시 ---


Training:   0%|          | 0/100 [00:00<?, ?it/s]

Early stopping triggered at epoch 19

--- [4] Vanilla Graph MLP (with Augmented Data)  학습 개시 ---


Training:   0%|          | 0/100 [00:00<?, ?it/s]

Early stopping triggered at epoch 17

--- [5] 최종 성능 검증 및 Symmetry Error 비교 ---
[Symmetric Model Evaluation Result - Permuted Data]
  Loss: 0.613802
  Accuracy: 66.18%
  Symmetry Error: 3.008628e-15

[Vanilla Model Evaluation Result - Permuted Data]
  Loss: 8.614733
  Accuracy: 53.77%
  Symmetry Error: 1.830772e+02

[Symmetric Model(augmented) Evaluation Result - Permuted Data]
  Loss: 0.607403
  Accuracy: 65.45%
  Symmetry Error: 3.746119e-15

[Vanilla Model(augmented) Evaluation Result - Permuted Data]
  Loss: 0.625395
  Accuracy: 63.75%
  Symmetry Error: 4.179037e-02


모든 Grid Search 실험 조건 완수 및 파일 저장 완료


In [34]:
result

{3: {500: {'symmetric': {'loss': 0.6321879625320435,
    'acc': 0.66,
    'sym_error': 3.33633059974327e-15},
   'vanilla': {'loss': 1.0809837579727173,
    'acc': 0.36,
    'sym_error': 0.3412201553583145},
   'symmetric (data augmented)': {'loss': 0.5570729970932007,
    'acc': 0.72,
    'sym_error': 4.354044074001101e-15},
   'vanilla (data augmented)': {'loss': 0.6934324502944946,
    'acc': 0.44,
    'sym_error': 0.006668172869831323}},
  1500: {'symmetric': {'loss': 0.6706274954477945,
    'acc': 0.6466666666666666,
    'sym_error': 5.658142168817121e-15},
   'vanilla': {'loss': 2.122624373435974,
    'acc': 0.56,
    'sym_error': 9.5591828028361},
   'symmetric (data augmented)': {'loss': 0.6480799587567647,
    'acc': 0.64,
    'sym_error': 5.746720865151692e-15},
   'vanilla (data augmented)': {'loss': 0.6599831430117289,
    'acc': 0.6333333333333333,
    'sym_error': 0.02248163359860579}},
  4110: {'symmetric': {'loss': 0.6036723190850585,
    'acc': 0.6593673965936739,
    

In [35]:

# 2. 데이터를 표 형태로 변환하기 위한 빈 리스트 생성
rows = []

# 3. 중첩 딕셔너리를 순회하며 행(Row) 데이터 추출
for n_block, data_dict in result.items():
    for data_size, model_dict in data_dict.items():
        for model_name, metrics in model_dict.items():
            # 각 행에 들어갈 데이터 정리
            row_data = {
                'Num_Blocks': n_block,
                'Dataset_Size': data_size,
                'Model_Type': model_name,
                'Test_Loss': metrics['loss'],
                'Test_Accuracy': metrics['acc'],
                'Symmetry_Error': metrics['sym_error']
            }
            rows.append(row_data)

# 4. DataFrame 생성
df_result = pd.DataFrame(rows)

# 5. 가독성을 위해 보기 좋게 정렬 (블록 수 -> 데이터 크기 -> 모델 순)
df_result = df_result.sort_values(by=['Num_Blocks', 'Dataset_Size', 'Model_Type']).reset_index(drop=True)

# 6. 결과 출력
# 소수점 출력을 깔끔하게 다듬기 위해 style 지정 (특히 Symmetry_Error는 지수 표기법 적용)
df_result.style.format({
    'Test_Loss': '{:.4f}',
    'Test_Accuracy': '{:.2%}',
    'Symmetry_Error': '{:.4e}'
})

,Num_Blocks,Dataset_Size,Model_Type,Test_Loss,Test_Accuracy,Symmetry_Error
0,3,500,symmetric,0.6322,66.00%,3.3363e-15
1,3,500,symmetric (data augmented),0.5571,72.00%,4.3540e-15
2,3,500,vanilla,1.0810,36.00%,3.4122e-01
3,3,500,vanilla (data augmented),0.6934,44.00%,6.6682e-03
4,3,1500,symmetric,0.6706,64.67%,5.6581e-15
5,3,1500,symmetric (data augmented),0.6481,64.00%,5.7467e-15
6,3,1500,vanilla,2.1226,56.00%,9.5592e+00
7,3,1500,vanilla (data augmented),0.6600,63.33%,2.2482e-02
8,3,4110,symmetric,0.6037,65.94%,9.2708e-15
9,3,4110,symmetric (data augmented),0.6064,66.91%,6.8118e-15


In [36]:
df_result.set_index(['Num_Blocks', "Dataset_Size"])

Model_Type  Test_Loss  Test_Accuracy  \
Num_Blocks Dataset_Size                                                         
3          500                            symmetric   0.632188       0.660000   
           500           symmetric (data augmented)   0.557073       0.720000   
           500                              vanilla   1.080984       0.360000   
           500             vanilla (data augmented)   0.693432       0.440000   
           1500                           symmetric   0.670627       0.646667   
           1500          symmetric (data augmented)   0.648080       0.640000   
           1500                             vanilla   2.122624       0.560000   
           1500            vanilla (data augmented)   0.659983       0.633333   
           4110                           symmetric   0.603672       0.659367   
           4110          symmetric (data augmented)   0.606422       0.669100   
           4110                             vanilla   3.549363       0.523114   
           4110            vanilla (data augmented)   0.611724       0.664234   
5          500                            symmetric   0.670516       0.580000   
           500           symmetric (data augmented)   0.690408       0.540000   
           500                              vanilla   2.550020       0.440000   
           500             vanilla (data augmented)   0.685601       0.460000   
           1500                           symmetric   0.635965       0.613333   
           1500          symmetric (data augmented)   0.645014       0.620000   
           1500                             vanilla   6.076594       0.526667   
           1500            vanilla (data augmented)   0.662032       0.560000   
           4110                           symmetric   0.604133       0.632603   
           4110          symmetric (data augmented)   0.593018       0.664234   
           4110                             vanilla   9.235891       0.520681   
           4110            vanilla (data augmented)   0.624151       0.613139   
7          500                            symmetric   0.628482       0.680000   
           500           symmetric (data augmented)   0.713839       0.400000   
           500                              vanilla   5.400650       0.340000   
           500             vanilla (data augmented)   0.725385       0.460000   
           1500                           symmetric   0.665540       0.566667   
           1500          symmetric (data augmented)   0.656798       0.660000   
           1500                             vanilla  10.658903       0.520000   
           1500            vanilla (data augmented)   0.747612       0.546667   
           4110                           symmetric   0.613802       0.661800   
           4110          symmetric (data augmented)   0.607403       0.654501   
           4110                             vanilla   8.614733       0.537713   
           4110            vanilla (data augmented)   0.625395       0.637470   

                         Symmetry_Error  
Num_Blocks Dataset_Size                  
3          500             3.336331e-15  
           500             4.354044e-15  
           500             3.412202e-01  
           500             6.668173e-03  
           1500            5.658142e-15  
           1500            5.746721e-15  
           1500            9.559183e+00  
           1500            2.248163e-02  
           4110            9.270812e-15  
           4110            6.811836e-15  
           4110            1.961513e+01  
           4110            2.808158e-02  
5          500             9.483246e-16  
           500             1.941835e-15  
           500             1.888843e+01  
           500             2.137975e-03  
           1500            2.305282e-15  
           1500            1.310696e-15  
           1500            3.670659e+01  
           1500            3.077700e-02  
           4110            3.857894e-1

In [37]:
df_result[df_result['Model_Type'] == 'symmetric']

,Num_Blocks,Dataset_Size,Model_Type,Test_Loss,Test_Accuracy,Symmetry_Error
0,3,500,symmetric,0.632188,0.660000,3.336331e-15
4,3,1500,symmetric,0.670627,0.646667,5.658142e-15
8,3,4110,symmetric,0.603672,0.659367,9.270812e-15
12,5,500,symmetric,0.670516,0.580000,9.483246e-16
16,5,1500,symmetric,0.635965,0.613333,2.305282e-15
20,5,4110,symmetric,0.604133,0.632603,3.857894e-15
24,7,500,symmetric,0.628482,0.680000,1.118078e-15
28,7,1500,symmetric,0.665540,0.566667,1.131576e-15
32,7,4110,symmetric,0.613802,0.661800,3.008628e-15


In [39]:
df_result[df_result['Model_Type'] == 'symmetric (data augmented)']

,Num_Blocks,Dataset_Size,Model_Type,Test_Loss,Test_Accuracy,Symmetry_Error
1,3,500,symmetric (data augmented),0.557073,0.720000,4.354044e-15
5,3,1500,symmetric (data augmented),0.648080,0.640000,5.746721e-15
9,3,4110,symmetric (data augmented),0.606422,0.669100,6.811836e-15
13,5,500,symmetric (data augmented),0.690408,0.540000,1.941835e-15
17,5,1500,symmetric (data augmented),0.645014,0.620000,1.310696e-15
21,5,4110,symmetric (data augmented),0.593018,0.664234,4.314071e-15
25,7,500,symmetric (data augmented),0.713839,0.400000,4.725108e-16
29,7,1500,symmetric (data augmented),0.656798,0.660000,1.933988e-15
33,7,4110,symmetric (data augmented),0.607403,0.654501,3.746119e-15


In [38]:
df_result[df_result['Model_Type'] == 'vanilla']

,Num_Blocks,Dataset_Size,Model_Type,Test_Loss,Test_Accuracy,Symmetry_Error
2,3,500,vanilla,1.080984,0.360000,0.341220
6,3,1500,vanilla,2.122624,0.560000,9.559183
10,3,4110,vanilla,3.549363,0.523114,19.615128
14,5,500,vanilla,2.550020,0.440000,18.888425
18,5,1500,vanilla,6.076594,0.526667,36.706591
22,5,4110,vanilla,9.235891,0.520681,161.414782
26,7,500,vanilla,5.400650,0.340000,25.397871
30,7,1500,vanilla,10.658903,0.520000,135.270168
34,7,4110,vanilla,8.614733,0.537713,183.077186


In [40]:
df_result[df_result['Model_Type'] == 'vanilla (data augmented)']

,Num_Blocks,Dataset_Size,Model_Type,Test_Loss,Test_Accuracy,Symmetry_Error
3,3,500,vanilla (data augmented),0.693432,0.440000,0.006668
7,3,1500,vanilla (data augmented),0.659983,0.633333,0.022482
11,3,4110,vanilla (data augmented),0.611724,0.664234,0.028082
15,5,500,vanilla (data augmented),0.685601,0.460000,0.002138
19,5,1500,vanilla (data augmented),0.662032,0.560000,0.030777
23,5,4110,vanilla (data augmented),0.624151,0.613139,0.052862
27,7,500,vanilla (data augmented),0.725385,0.460000,0.021860
31,7,1500,vanilla (data augmented),0.747612,0.546667,0.078554
35,7,4110,vanilla (data augmented),0.625395,0.637470,0.041790
